# Import Library

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi Path Dataset
DATASET_PATH = '../data/processed/dataset_training_final_FIXED.csv'

print("Library berhasil diimport.")

# Memuat Dataset

In [ ]:
try:
    df = pd.read_csv(DATASET_PATH)
    print(f"Dataset dimuat: {len(df)} baris.")
    print("Kolom tersedia:", list(df.columns))
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di {DATASET_PATH}")
    print("Pastikan Anda sudah menjalankan notebook 04_feature_extraction.ipynb")

# Cek distribusi label
print("\nDistribusi Label:")
print(df['label'].value_counts(normalize=True))

# Persiapan Data (Split Train-Test)"

In [ ]:
# Definisi Target
target_col = 'label'

# Memisahkan Fitur dan Target
X = df.drop(columns=[target_col, 'query', 'tafsir_text']) # Hapus kolom teks, hanya sisakan angka
y = df[target_col]

# Split Data (80:20)
# Stratify digunakan agar proporsi label positif/negatif seimbang di train dan test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Definisi Konfigurasi Fitur

In [ ]:
# Grup Fitur
features_lexical = ['bm25_score', 'jaccard_score', 'overlap_score']
features_semantic = ['sbert_sim']
features_hybrid = features_lexical + features_semantic

print("Konfigurasi Fitur:")
print(f"1. Lexical Only : {features_lexical}")
print(f"2. Semantic Only: {features_semantic}")
print(f"3. Full Hybrid  : {features_hybrid}")

# Fungsi Training dan Evaluasi

In [ ]:
def train_and_evaluate(features, feature_group_name):
    print(f"\n Training: {feature_group_name} ")
    # Filter dataset hanya untuk fitur yang dipilih
    X_tr_subset = X_train[features]
    X_te_subset = X_test[features]
    
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1
    )
    
    model.fit(X_tr_subset, y_train)
    y_pred_proba = model.predict_proba(X_te_subset)[:, 1]
    map_score = average_precision_score(y_test, y_pred_proba)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    print(f"Hasil {feature_group_name}: MAP = {map_score:.4f} | AUC = {auc_score:.4f}")
    
    return map_score

# Eksekusi Studi Ablasi

In [ ]:
score_lexical = train_and_evaluate(features_lexical, "Lexical Only")
score_semantic = train_and_evaluate(features_semantic, "Semantic Only")
score_hybrid = train_and_evaluate(features_hybrid, "Full Hybrid")

# Analisis Hasil

In [ ]:
# Hitung Improvement (%)
improvement = ((score_hybrid - score_lexical) / score_lexical) * 100

# Buat DataFrame Hasil Akhir
results_df = pd.DataFrame({
    'Model Configuration': ['Lexical Only', 'Semantic Only', 'Full Hybrid'],
    'MAP Score': [score_lexical, score_semantic, score_hybrid]
})

print("HASIL AKHIR SKENARIO 3 (ABLATION STUDY)")
print(results_df)
print(f"Baseline (Lexical): {score_lexical:.4f}")
print(f"Full Hybrid       : {score_hybrid:.4f}")
print(f"Improvement       : +{improvement:.2f}%")